In [67]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
import os
import random
import json
import time
import math
import zipfile
import shutil
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from collections import Counter

In [68]:


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

In [69]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Seq2SeqConfig:
    vocab_size: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    n_layers: int = 8

# ---------------------------------------------------------
# 1. Attention Modules
# ---------------------------------------------------------
class EncoderSelfAttention(nn.Module):
    """Bidirectional self-attention for the encoder (no causal mask)."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CausalSelfAttention(nn.Module):
    """Causal self-attention for the decoder (looks only at past tokens)."""
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CrossAttention(nn.Module):
    """Cross-attention: Query from Decoder, Key/Value from Encoder."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        B_c, T_c, C_c = context.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.kv_proj(context).chunk(2, dim=-1)
        k, v = [t.view(B_c, T_c, self.n_heads, self.d_head).transpose(1, 2) for t in kv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

# ---------------------------------------------------------
# 2. Feed-Forward & Standard Blocks
# ---------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


class EncoderBlock(nn.Module):
    """Standard bidirectional Transformer encoder block."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = EncoderSelfAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    """Standard causal Transformer decoder block with cross-attention."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout)
        
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), encoder_out)
        x = x + self.ffn(self.ln3(x))
        return x





# ---------------------------------------------------------
# 5. Top-Level Hourglass Seq2Seq Model
# ---------------------------------------------------------
class StandardEncoderDecoder(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)

        self.encoder_layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.encoder_ln = nn.LayerNorm(cfg.d_model)

        self.decoder_layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.decoder_ln = nn.LayerNorm(cfg.d_model)

        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def get_positional_embeddings(self, seq_len, device):
        return self.pos_emb(torch.arange(seq_len, device=device).unsqueeze(0))

    def encode(self, enc_idx):
        x = self.tok_emb(enc_idx) + self.get_positional_embeddings(enc_idx.size(1), enc_idx.device)
        for block in self.encoder_layers:
            x = block(x)
        return self.encoder_ln(x)

    def decode(self, dec_idx, encoder_out):
        x = self.tok_emb(dec_idx) + self.get_positional_embeddings(dec_idx.size(1), dec_idx.device)
        for block in self.decoder_layers:
            x = block(x, encoder_out)
        return self.lm_head(self.decoder_ln(x))

    def forward(self, enc_idx, dec_idx):
        return self.decode(dec_idx, self.encode(enc_idx))

In [70]:
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 2000
    max_seq_len: int = 256
    dropout: float = 0.15
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


@dataclass
class EncDecConfig(Seq2SeqConfig):
    enc_len: int = None      # tokens the encoder reads (prompt side)
    dec_len: int = None      # tokens the decoder learns to produce (continuation side)


class EncDecLM(nn.Module):
    def __init__(self, cfg: EncDecConfig):
        super().__init__()
        assert None not in (cfg.enc_len, cfg.dec_len), "set enc_len and dec_len in the config"
        self.cfg = cfg
        self.core = StandardEncoderDecoder(cfg)

    def encode(self, enc_idx):
        return self.core.encode(enc_idx)

    def decode(self, dec_idx, encoder_out):
        return self.core.decode(dec_idx, encoder_out)

    def forward(self, enc_idx, dec_idx):
        return self.core(enc_idx, dec_idx)

In [71]:
# Cell A

import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudraalt/phase-1-c-6-simple-encoder-decoder"   # fix to what find showed
device = torch.device("cuda")

tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/bpe_tokenizer.pkl", "rb"))   # c-6's own tokenizer (vocab 11022)
ck = torch.load(f"{ROOT}/cs781_phase1_full/model_encdec/model_final.pt", map_location="cpu", weights_only=False)
cfg = EncDecConfig(**ck["config"])
lm = EncDecLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| n_layers", cfg.n_layers, "| max_seq_len", cfg.max_seq_len)

<All keys matched successfully>
params 55.13008 M | vocab 11022 | d_model 560 | n_layers 8 | max_seq_len 1536


In [72]:
# Cell B

def features(idx):                        # idx: (B, T) LongTensor, T <= 1536
    c = lm.core
    T = idx.size(1)
    assert T <= c.cfg.max_seq_len         # learned position table has 1536 rows
    x = c.tok_emb(idx) + c.get_positional_embeddings(T, idx.device)
    for blk in c.encoder_layers:          # 8 bidirectional EncoderBlocks
        x = blk(x)
    return c.encoder_ln(x)                # (B, T, 560)

# Sanity check: same as the model's own encoder
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f, ref = features(idx), lm.encode(idx)
print("max diff", (f - ref).abs().max().item(), tuple(f.shape))   # expect 0.0, (1, 64, 560)

max diff 0.0 (1, 64, 560)


In [73]:
# Cell C

import json
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

# Get your Hugging Face token from Kaggle Secrets
hf_token = UserSecretsClient().get_secret("HF_TOKEN")

def get(name):
    p = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename=f"Phase_2_LNER/{name}",
        repo_type="dataset",
        token=hf_token
    )

    if p.endswith(".json"):
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    else:
        with open(p, "r", encoding="utf-8") as f:
            return f.read().split()

f1_text, f1_gold = get("Train/fold1-text.json"), get("Train/fold1-gold.json")
f2_text, f2_gold = get("Train/fold2-text.json"), get("Train/fold2-gold.json")
test_text = get("Test/fold3-text.json")
LABELS = get("Train/labels.txt")

print(len(f1_text), len(f2_text), len(test_text), LABELS)

k = next(iter(f1_gold))
print(k, f1_gold[k][:3])  # Check annotation format

35 35 35 ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
115651329 [[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011'], [276, 308, 'COURT', 'High Court of Punjab and Haryana']]


In [74]:
# Cell D

SPLIT = re.compile(r'\s*\S+|\s+')

def encode_with_offsets(text):
    ids, offs = [], []
    for m in SPLIT.finditer(text):
        chunk, cs = m.group(), m.start()
        b2c = []
        for ci, ch in enumerate(chunk):
            b2c += [ci] * len(ch.encode("utf-8"))
        b = 0
        for t in tok.encode(chunk):
            n = len(tok.id_to_bytes[t])
            s, e = cs + b2c[b], cs + b2c[b + n - 1] + 1
            ids.append(t); offs.append((s, e)); b += n
    return ids, offs

def content_start(text, s, e):            # skip the leading space a token carries
    while s < e and text[s].isspace(): s += 1
    return s

def spans(gold_doc):                      # -> list of (start, end, label)
    return [(int(g[0]), int(g[1]), g[2]) for g in gold_doc]

# Check 1: token ids match the Phase 1 tokenizer exactly
t = f1_text[k]; ids, offs = encode_with_offsets(t)
assert ids == tok.encode(t), "chunked encoding differs from tok.encode"

# Check 2: gold -> token BIO -> back to spans
def to_bio(text, offs, ents):
    tags = ["O"] * len(offs)
    for s, e, lab in ents:
        first = True
        for i, (ts, te) in enumerate(offs):
            cs_ = content_start(text, ts, te)
            if cs_ < e and te > s and cs_ < te:      # token overlaps the entity
                tags[i] = ("B-" if first else "I-") + lab; first = False
    return tags

def from_bio(text, offs, tags):
    out, cur = [], None
    for (ts, te), tg in zip(offs, tags):
        if tg.startswith("B-") or (tg.startswith("I-") and (cur is None or cur[2] != tg[2:])):
            if cur: out.append(tuple(cur))
            cur = [content_start(text, ts, te), te, tg[2:]]
        elif tg.startswith("I-"):
            cur[1] = te
        else:
            if cur: out.append(tuple(cur)); cur = None
    if cur: out.append(tuple(cur))
    return out

hit = tot = 0; bad = []
for texts, golds in [(f1_text, f1_gold), (f2_text, f2_gold)]:
    for d in golds:
        text = texts[d]; _, offs = encode_with_offsets(text)
        g = set(spans(golds[d])); r = set(from_bio(text, offs, to_bio(text, offs, g)))
        hit += len(g & r); tot += len(g)
        bad += [(text[s:e], lab, d) for s, e, lab in g - r][:3]
print(f"round-trip exact: {hit}/{tot} = {hit/tot:.2%}"); print(bad[:15])

round-trip exact: 6135/8393 = 73.10%
[('Ramesh Kumar', 'WIT', '115651329'), ('High Court', 'COURT', '115651329'), ('V. Gopala Gowda', 'JUDGE', '115651329'), ('14.05.2010', 'DATE', '37849282'), ('12.12.2005', 'DATE', '37849282'), ('31.12.2003', 'DATE', '37849282'), ('TP Act', 'STAT', '975074'), ('Civil Court', 'COURT', '975074'), ('High Court', 'COURT', '975074'), ('Ramveer', 'APP', '189525449'), ('IPC', 'STAT', '189525449'), ('Surendra', 'APP', '189525449'), ('1.9.1999', 'DATE', '736324'), ('High Court', 'COURT', '736324'), ('O.A. No. 103 of 1999', 'CASENO', '736324')]


In [75]:
# =====================================================================================
# CELL E0 — shared plumbing for the c-6 Phase 2 experiments (LoRA / LORMA / KD)
# Run once after Cells 0, A, B, C, D of c6-next-cells.md. Then run ONE technique cell:
#   cellE1_lora_c6.py, cellE2_lorma_c6.py or cellE3_kd_student_c6.py
# Needs from earlier cells: lm, tok, cfg, device, features, LABELS,
#   f1_text, f1_gold, f2_text, f2_gold, test_text, encode_with_offsets, to_bio, from_bio, spans
#
# Nothing in this cell is a model design or a hyperparameter choice. It only does:
# tags, windows, batching, the c-6 encoder forward path, scoring, the training loop,
# prediction and packaging.
# =====================================================================================
import math, time, json, zipfile, random
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm   # every loop below that takes time shows a progress bar

# -------------------------------------------------------------------------------------
# Tags: O + B-/I- for each of the 12 labels -> 25 tags
# -------------------------------------------------------------------------------------
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
N_TAGS = len(TAGS)
IGNORE = -100          # positions with no loss
FEAT_DIM = cfg.d_model # 560 for c-6

# The decoder is never used for NER; dropping it saves ~29M params of GPU memory.
# tok_emb stays (the LM head shares its weight).
lm.core.decoder_layers = None
lm.core.decoder_ln = None
torch.cuda.empty_cache()


# -------------------------------------------------------------------------------------
# c-6 encoder path (identical to Cell B features(), but on any backbone object)
# -------------------------------------------------------------------------------------
def encoder_forward(backbone, idx):
    c = backbone.core
    assert idx.size(1) <= c.cfg.max_seq_len, "window longer than 1536"
    x = c.tok_emb(idx) + c.get_positional_embeddings(idx.size(1), idx.device)
    for blk in c.encoder_layers:
        x = blk(x)
    return c.encoder_ln(x)                      # (B, T, 560)


def encoder_linears(backbone, which):
    """Yield (parent, attr, nn.Linear) for the chosen linears in every encoder block.
    which ⊆ {"qkv_proj", "out_proj", "ffn_in", "ffn_out"}.
    c-6 encoder attention has ONE fused qkv_proj (560 -> 1680), not q/k/v_proj."""
    for blk in backbone.core.encoder_layers:
        for w in which:
            if w in ("qkv_proj", "out_proj"):
                parent, attr = blk.self_attn, w
            elif w == "ffn_in":
                parent, attr = blk.ffn.net, "0"     # 560 -> d_ff
            elif w == "ffn_out":
                parent, attr = blk.ffn.net, "2"     # d_ff -> 560
            else:
                raise ValueError(w)
            yield parent, attr, getattr(parent, attr)


def wrap_linears(backbone, which, make_wrapper):
    """Replace each chosen linear by make_wrapper(linear). Returns how many were wrapped."""
    todo = list(encoder_linears(backbone, which))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)


class NERModel(nn.Module):
    """backbone (c-6 encoder path) -> head -> per-token tag logits."""
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone, self.head = backbone, head
    def forward(self, idx):
        return self.head(encoder_forward(self.backbone, idx))   # (B, T, 25)


def identity_probe():
    """64 real tokens from fold1, for the 'adapter = identity at init' check."""
    t = next(iter(f1_text.values()))
    return torch.tensor([encode_with_offsets(t)[0][:64]], device=device)


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# Windows. The encoder has no padding mask, so windows are never padded:
#   - a document of n <= WINDOW tokens is one window of length n;
#   - longer documents get full-length windows every STRIDE tokens, the last one
#     placed so it ends exactly at the document end.
# Each token is "owned" by the window where it is furthest from an edge
# (most context on both sides); predictions are taken from the owner only.
# -------------------------------------------------------------------------------------
def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]


def ownership(wins, n):
    best_w, best_d = [0] * n, [-1] * n
    for w, (s, e) in enumerate(wins):
        for i in range(s, e):
            d = min(i - s, e - 1 - i)
            if d > best_d[i]:
                best_d[i], best_w[i] = d, w
    return best_w


def build_examples(texts, golds, window, stride, label_owned_only, soft=None):
    """One example per window.
    golds: dict or None (test).  soft: optional {doc: (n_tok, 25) tensor, mask (n_tok,)} for KD.
    label_owned_only: True -> a window gets labels only on the tokens it owns (each token
    trained once); False -> every token in the window is labelled (overlap trained twice)."""
    examples, cache = [], {}
    for doc_id, text in tqdm(texts.items(), desc="tokenising + windowing", leave=False):
        ids, offs = encode_with_offsets(text)
        n = len(ids)
        cache[doc_id] = (text, offs)
        if n == 0:                      # empty document: no windows, nothing to tag
            continue
        tag_ids = None
        if golds is not None:
            tag_ids = [TAG2ID[t] for t in to_bio(text, offs, spans(golds.get(doc_id, [])))]
        wins = make_windows(n, window, stride)
        owner = ownership(wins, n)
        for w, (s, e) in enumerate(wins):
            own = [owner[i] == w for i in range(s, e)]
            ex = {"doc": doc_id, "start": s, "ids": ids[s:e], "own": own, "labels": None}
            if tag_ids is not None:
                ex["labels"] = [t if (o or not label_owned_only) else IGNORE
                                for t, o in zip(tag_ids[s:e], own)]
            if soft is not None and doc_id in soft:
                ex["soft"] = soft[doc_id]["logits"][s:e]
                ex["soft_mask"] = soft[doc_id]["mask"][s:e]
            examples.append(ex)
    return examples, cache


def make_batches(examples, batch_size, shuffle):
    """Batches of windows with IDENTICAL length (no padding needed)."""
    by_len = defaultdict(list)
    for ex in examples:
        by_len[len(ex["ids"])].append(ex)
    out = []
    for exs in by_len.values():
        if shuffle:
            random.shuffle(exs)
        out += [exs[i:i + batch_size] for i in range(0, len(exs), batch_size)]
    if shuffle:
        random.shuffle(out)
    return out


def to_tensors(batch):
    b = {"ids": torch.tensor([ex["ids"] for ex in batch], device=device)}
    if batch[0]["labels"] is not None:
        b["labels"] = torch.tensor([ex["labels"] for ex in batch], device=device)
    if "soft" in batch[0]:
        b["soft"] = torch.stack([ex["soft"] for ex in batch]).to(device).float()       # (B, T, 25) teacher logits
        b["soft_mask"] = torch.stack([ex["soft_mask"] for ex in batch]).to(device)     # (B, T) bool
    return b


def ce_loss(logits, labels, class_weights=None):
    """Plain token cross-entropy on gold tags (ignores IGNORE positions)."""
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1),
                           weight=class_weights, ignore_index=IGNORE)


# -------------------------------------------------------------------------------------
# Prediction + strict scoring
# -------------------------------------------------------------------------------------
TRAIL_CHARS = ",.;:"   # punctuation that BPE glues onto an entity's last token but gold spans exclude;
                       # set to "" to switch off. Verify it with the oracle check (see thread).

def decode_spans(text, offs, tags):
    out = []
    for s, e, l in from_bio(text, offs, tags):
        while TRAIL_CHARS and e > s and text[e - 1] in TRAIL_CHARS:
            e -= 1
        if e > s:
            out.append((s, e, l))
    return out


@torch.no_grad()
def predict(model, examples, cache, batch_size):
    """{doc_id: [(start_char, end_char, label), ...]} using each token's owner window."""
    model.eval()
    doc_tags = {d: ["O"] * len(cache[d][1]) for d in cache}
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="predict", leave=False):
        with torch.autocast("cuda", dtype=torch.float16):
            pred = model(to_tensors(batch)["ids"]).argmax(-1).cpu()
        for b, ex in enumerate(batch):
            for j, o in enumerate(ex["own"]):
                if o:
                    doc_tags[ex["doc"]][ex["start"] + j] = TAGS[pred[b, j]]
    return {d: decode_spans(cache[d][0], cache[d][1], doc_tags[d]) for d in cache}


def strict_macro_f1(pred, gold):
    """Exact (start, end, label) match; per-label P/R/F1, macro over labels seen in gold or
    predictions. Cross-check with the official script before trusting small differences."""
    tp = {l: 0 for l in LABELS}; fp = dict(tp); fn = dict(tp)
    for d in gold:
        g, p = set(spans(gold[d])), set(pred.get(d, []))
        for s, e, l in p & g: tp[l] += 1
        for s, e, l in p - g: fp[l] = fp.get(l, 0) + 1
        for s, e, l in g - p: fn[l] += 1
    used = [l for l in LABELS if tp[l] + fp[l] + fn[l] > 0]
    per = {}
    for l in used:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = (P, R, 2 * P * R / (P + R) if P + R else 0.0)
    return (sum(v[2] for v in per.values()) / len(per) if per else 0.0), per


# -------------------------------------------------------------------------------------
# Training loop. loss_fn(logits, batch_tensors) is supplied by the technique cell,
# so KD can use the teacher's soft labels and LoRA/LORMA plain CE.
# lr_lambda: optional step -> multiplier (your schedule), or None for constant LR.
# -------------------------------------------------------------------------------------
def train(model, train_ex, val_ex, val_cache, val_gold, loss_fn, *, lr, epochs, batch_size,
          weight_decay=0.0, grad_clip=1.0, lr_lambda=None, save_path="best_trainable.pt"):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    t0, best_f1, best_per, log = time.time(), -1.0, {}, []
    for epoch in range(epochs):
        model.train(); tot, nb = 0.0, 0
        pbar = tqdm(make_batches(train_ex, batch_size, shuffle=True), desc=f"epoch {epoch + 1}/{epochs}")
        for batch in pbar:
            b = to_tensors(batch)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["ids"])
            loss = loss_fn(logits.float(), b)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched: sched.step()
            tot += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{tot / nb:.4f}")
        f1, per = strict_macro_f1(predict(model, val_ex, val_cache, batch_size), val_gold)
        log.append({"epoch": epoch, "train_loss": tot / max(1, nb), "val_macro_f1": f1})
        print(log[-1])
        if f1 > best_f1:
            best_f1, best_per = f1, per
            torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, save_path)
    print(f"best val strict macro-F1 {best_f1:.4f} | train time {(time.time()-t0)/60:.1f} min | "
          f"peak GPU mem {torch.cuda.max_memory_allocated()/2**30:.2f} GB")
    for l, (P, R, F1) in best_per.items():
        print(f"  {l:12s} P {P:.3f} R {R:.3f} F1 {F1:.3f}")
    return log, best_f1, best_per


@torch.no_grad()
def time_inference(model, examples, cache, batch_size):
    """Wall-clock seconds to predict a set (for the efficiency table in the report)."""
    torch.cuda.synchronize(); t = time.time()
    predict(model, examples, cache, batch_size)
    torch.cuda.synchronize(); return time.time() - t


def write_submission(model, window, stride, batch_size, best_path=None, out_dir="/kaggle/working"):
    if best_path:
        model.load_state_dict(torch.load(best_path), strict=False)
    test_ex, test_cache = build_examples(test_text, None, window, stride, True)
    pred = predict(model, test_ex, test_cache, batch_size)
    out = {d: [[s, e, l] for s, e, l in pred.get(d, [])] for d in test_text}
    assert set(out) == set(test_text), "every fold3 ID, no extras"
    with open(f"{out_dir}/prediction.json", "w") as f:
        json.dump(out, f)
    with zipfile.ZipFile(f"{out_dir}/submission.zip", "w") as z:
        z.write(f"{out_dir}/prediction.json", arcname="prediction.json")
    print("entities predicted:", sum(len(v) for v in out.values()), f"-> {out_dir}/submission.zip")


In [76]:
# =====================================================================================
# CELL E1 — LoRA on the c-6 encoder for Legal NER (template)
# Run after Cells 0, A-D and E0 (cellE0_common_c6.py).
#
# YOUR WORK (course rule: no LLM-written architecture or hyperparameters):
#   [H] hyperparameters below
#   [L] LoRALinear: the low-rank update itself (parameters, init, forward)  — ref [3] Hu et al. 2021
#   [N] the NER head
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Hyperparameters — all yours. None = not set; the assert stops you.
# -------------------------------------------------------------------------------------
WINDOW       = 200   # tokens per window, <= 1536
STRIDE       = 10   # step between window starts (< WINDOW); WINDOW - STRIDE = overlap
LABEL_OWNED_ONLY = True   # True: each token trained once; False: overlap tokens trained in every window
LORA_RANK    = 200   # r
LORA_ALPHA   = 50   # scaling numerator (update is scaled by alpha / r in the paper)
LORA_TARGETS = ("ffn_out",)   # subset of ("qkv_proj", "out_proj", "ffn_in", "ffn_out") — which linears get LoRA
LR           = 1e-4
EPOCHS       = 10
BATCH_SIZE   = 64   # windows per step
WEIGHT_DECAY = 0.0
LR_LAMBDA    = None   # optional: step -> LR multiplier (warm-up / decay), or None for constant
CLASS_WEIGHTS = None  # optional: torch.tensor([...25...], device=device) for rare labels
SEED         = 42
for _n in ["WINDOW", "STRIDE", "LABEL_OWNED_ONLY", "LORA_RANK", "LORA_ALPHA", "LORA_TARGETS", "LR", "EPOCHS", "BATCH_SIZE"]:
    assert globals()[_n] is not None, f"set {_n}"
assert STRIDE < WINDOW <= cfg.max_seq_len
random.seed(SEED); torch.manual_seed(SEED)


# -------------------------------------------------------------------------------------
# [L] LoRA layer — TODO
#   Wraps one frozen nn.Linear (y = x W^T + b). Add the trainable low-rank update.
#   Requirement checked below: at initialisation the output must equal self.base(x).
# -------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.rank, self.alpha = rank, alpha
        self.A = nn.Linear(self.in_features, rank, bias=False)
        self.B = nn.Linear(rank, self.out_features, bias=False)

        nn.init.normal_(self.A.weight, mean=0.0, std=1.0)
        nn.init.zeros_(self.B.weight)
        
        self.scaling = alpha / rank
    def forward(self, x):
        # TODO: base output + scaled low-rank update
        final = self.base(x) + self.scaling * self.B(self.A(x))
        return final


# -------------------------------------------------------------------------------------
# Freeze c-6, wrap the chosen linears, check identity at init
# -------------------------------------------------------------------------------------
lm.eval()
probe = identity_probe()
with torch.no_grad():
    ref = features(probe).float()

for p in lm.parameters():
    p.requires_grad = False
n_wrapped = wrap_linears(lm, LORA_TARGETS, lambda lin: LoRALinear(lin, LORA_RANK, LORA_ALPHA).to(device))

with torch.no_grad():
    diff = (features(probe).float() - ref).abs().max().item()
print(f"wrapped {n_wrapped} linears | LoRA identity-at-init max |diff| = {diff:.2e} (should be ~0)")


# -------------------------------------------------------------------------------------
# [N] Head — TODO: map FEAT_DIM (560) -> N_TAGS (25). Your design.
# -------------------------------------------------------------------------------------
hdim = 200
head = head = nn.Sequential(
    nn.Linear(FEAT_DIM, hdim),
    nn.ReLU(),
    nn.Linear(hdim, N_TAGS)
)   # TODO e.g. an nn.Module of your own
assert head is not None, "define the head"
model = NERModel(lm, head).to(device)
count_params(model)


# -------------------------------------------------------------------------------------
# Data, training, scoring (plumbing from E0)
# -------------------------------------------------------------------------------------
train_ex, _ = build_examples(f1_text, f1_gold, WINDOW, STRIDE, LABEL_OWNED_ONLY)
val_ex, val_cache = build_examples(f2_text, f2_gold, WINDOW, STRIDE, True)
print(f"train windows {len(train_ex)} | val windows {len(val_ex)}")

log, best_f1, per = train(
    model, train_ex, val_ex, val_cache, f2_gold,
    loss_fn=lambda logits, b: ce_loss(logits, b["labels"], CLASS_WEIGHTS),
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
    lr_lambda=LR_LAMBDA, save_path="/kaggle/working/c6_lora_best_trainable.pt")
print(f"fold2 inference time {time_inference(model, val_ex, val_cache, BATCH_SIZE):.1f} s")

# Predict fold3 with the best weights and package
write_submission(model, WINDOW, STRIDE, BATCH_SIZE, best_path="/kaggle/working/c6_lora_best_trainable.pt")


wrapped 8 linears | LoRA identity-at-init max |diff| = 0.00e+00 (should be ~0)
trainable 2.613M / total 28.67M (9.11%)


tokenising + windowing:   0%|          | 0/35 [00:00<?, ?it/s]

tokenising + windowing:   0%|          | 0/35 [00:00<?, ?it/s]

train windows 24270 | val windows 24309


epoch 1/10:   0%|          | 0/380 [00:00<?, ?it/s]

predict:   0%|          | 0/380 [00:00<?, ?it/s]

... (29 outputs left out)

fold2 inference time 36.7 s


tokenising + windowing:   0%|          | 0/35 [00:00<?, ?it/s]

predict:   0%|          | 0/409 [00:00<?, ?it/s]

entities predicted: 5707 -> /kaggle/working/submission.zip
